In [2]:
import pandas as pd 



In [4]:
data = pd.read_excel('./data/Wind_Turbine_Database_en.xlsx')
print(data.head())

  Province_Territory          Project Name  Total Project Capacity (MW)  \
0            Alberta  Ardenville Wind Farm                         69.0   
1            Alberta  Ardenville Wind Farm                         69.0   
2            Alberta  Ardenville Wind Farm                         69.0   
3            Alberta  Ardenville Wind Farm                         69.0   
4            Alberta  Ardenville Wind Farm                         69.0   

  Turbine Identifier  Turbine Number  Number of Turbines in Project  \
0               AWF1               1                             23   
1               AWF2               2                             23   
2               AWF3               3                             23   
3               AWF4               4                             23   
4               AWF5               5                             23   

  Turbine Rated Capacity (kW)  Rotor Diameter (m) Hub Height (m) Manufacturer  \
0                        3000            

In [5]:
# Filter for Ontario province
ontario_data = data[data['Province_Territory'] == 'Ontario']

print(f"Total records: {len(data)}")
print(f"Ontario records: {len(ontario_data)}")
print(ontario_data.head())

Total records: 7841
Ontario records: 2712
     Province_Territory                 Project Name  \
2736            Ontario  Adelaide Wind Energy Centre   
2737            Ontario  Adelaide Wind Energy Centre   
2738            Ontario  Adelaide Wind Energy Centre   
2739            Ontario  Adelaide Wind Energy Centre   
2740            Ontario  Adelaide Wind Energy Centre   

      Total Project Capacity (MW) Turbine Identifier  Turbine Number  \
2736                        59.94               AWE1               1   
2737                        59.94               AWE2               2   
2738                        59.94               AWE3               3   
2739                        59.94               AWE4               4   
2740                        59.94               AWE5               5   

      Number of Turbines in Project Turbine Rated Capacity (kW)  \
2736                             37                        1620   
2737                             37                   

In [ ]:
from ipywidgets import IntSlider, Output, VBox
from IPython.display import display

# Get unique commissioning years and sort them
def extract_year(val):
    """Extract year from commissioning value (handles ranges like '2006/2008')"""
    if pd.isna(val) or str(val) == 'nan':
        return None
    try:
        # Convert to string and handle ranges
        val_str = str(val).strip()
        if '/' in val_str:
            val_str = val_str.split('/')[0]  # Take first year from range
        return int(float(val_str))
    except:
        return None

# Extract years and remove None values
years = [extract_year(val) for val in ontario_data['Commissioning']]
years = [y for y in years if y is not None]
years = sorted(set(years))  # Remove duplicates and sort

min_year = min(years)
max_year = max(years)

# Create slider widget positioned above the map
year_slider = IntSlider(
    value=max_year,
    min=min_year,
    max=max_year,
    step=1,
    description='Year:',
    continuous_update=False
)

# Output widget to display the map
map_output = Output()

def update_map(change):
    map_output.clear_output(wait=True)
    
    with map_output:
        selected_year = year_slider.value
        
        # Filter data: show turbines that existed by selected year (built in or before that year)
        mask = ontario_data['Commissioning'].apply(lambda x: extract_year(x) is not None and extract_year(x) <= selected_year)
        filtered_data = ontario_data[mask]
        
        # Create a new map
        ontario_map = folium.Map(
            location=[51.2538, -85.3232],
            zoom_start=6,
            tiles='CartoDB positron'
        )
        
        # Add markers with clustering
        marker_cluster = MarkerCluster().add_to(ontario_map)
        
        for idx, row in filtered_data.iterrows():
            # Create popup with all columns
            popup_text = "<br>".join([f"<b>{col}</b>: {row[col]}" for col in filtered_data.columns])
            popup = folium.Popup(popup_text, max_width=300)
            
            folium.Marker(
                location=[row['Latitude'], row['Longitude']],
                popup=popup
            ).add_to(marker_cluster)
        
        # Save the map
        ontario_map.save('ontario_turbines_map.html')
        
        # Display count and map
        print(f"Showing {len(filtered_data)} turbines that existed by {selected_year}")
        display(ontario_map)

# Attach the callback
year_slider.observe(update_map, names='value')

# Display slider above map
display(year_slider)

# Show initial map
update_map(None)

Dropdown(description='Select Year:', options=(('All Years', None), ('1995', 1995), ('2001', 2001), ('2002', 20…